In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
results_df = pd.read_csv("../data/raw/race_history/results.csv")
lap_times_df = pd.read_csv("../data/raw/race_history/lap_times.csv")

results_df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [3]:
mask = results_df.astype(str).apply(lambda col: col.str.contains(r'\\N', na=False)).any(axis=1)
print(f"Rows containing \\N in results.csv: {mask.sum()}")

Rows containing \N in results.csv: 22494


In [4]:
print(results_df.dtypes)

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number                 str
grid                 int64
position               str
positionText           str
positionOrder        int64
points             float64
laps                 int64
time                   str
milliseconds           str
fastestLap             str
rank                   str
fastestLapTime         str
fastestLapSpeed        str
statusId             int64
dtype: object


In [5]:
from src.transform import clean_nulls

results_clean = clean_nulls(results_df)

print("BEFORE cleaning:")
print(results_df.isnull().sum())
print("\nAFTER cleaning:")
print(results_clean.isnull().sum())

BEFORE cleaning:
resultId           0
raceId             0
driverId           0
constructorId      0
number             0
grid               0
position           0
positionText       0
positionOrder      0
points             0
laps               0
time               0
milliseconds       0
fastestLap         0
rank               0
fastestLapTime     0
fastestLapSpeed    0
statusId           0
dtype: int64

AFTER cleaning:
resultId               0
raceId                 0
driverId               0
constructorId          0
number                 6
grid                   0
position           10953
positionText           0
positionOrder          0
points                 0
laps                   0
time               19079
milliseconds       19079
fastestLap         18507
rank               18249
fastestLapTime     18507
fastestLapSpeed    18507
statusId               0
dtype: int64


In [6]:
from src.transform import parse_lap_time

results_clean['fastest_lap_ms'] = results_clean['fastestLapTime'].apply(parse_lap_time)

results_clean[['fastestLapTime', 'fastest_lap_ms']].dropna().head(10)

,fastestLapTime,fastest_lap_ms
0,1:27.452,87452.0
1,1:27.739,87739.0
2,1:28.090,88090.0
3,1:28.603,88603.0
4,1:27.418,87418.0
5,1:29.639,89639.0
6,1:29.534,89534.0
7,1:27.903,87903.0
8,1:28.753,88753.0
9,1:29.558,89558.0


In [7]:
sample = results_clean[['fastestLapTime', 'fastest_lap_ms']].dropna().iloc[0]
print(f"Original: {sample['fastestLapTime']}")
print(f"Converted: {sample['fastest_lap_ms']} ms")
print(f"Converted back to minutes: {sample['fastest_lap_ms'] / 60000:.3f} minutes")

Original: 1:27.452
Converted: 87452.0 ms
Converted back to minutes: 1.458 minutes


In [8]:
import os

race_history_dir = "../data/raw/race_history"
all_files = [f for f in os.listdir(race_history_dir) if f.endswith(".csv")]
print(all_files)

['circuits.csv', 'constructors.csv', 'constructor_results.csv', 'constructor_standings.csv', 'drivers.csv', 'driver_standings.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'seasons.csv', 'sprint_results.csv', 'status.csv']


In [9]:
[f for f in os.listdir(race_history_dir) if f.endswith(".csv")]

['circuits.csv',
 'constructors.csv',
 'constructor_results.csv',
 'constructor_standings.csv',
 'drivers.csv',
 'driver_standings.csv',
 'lap_times.csv',
 'pit_stops.csv',
 'qualifying.csv',
 'races.csv',
 'results.csv',
 'seasons.csv',
 'sprint_results.csv',
 'status.csv']

In [10]:
all_files = []
for f in os.listdir(race_history_dir):
    if f.endswith(".csv"):
        all_files.append(f)

In [3]:
processed_dir = "../data/processed/race_history"
os.makedirs(processed_dir, exist_ok=True)

cleaning_summary = []

for filename in all_files:
    raw_path = os.path.join(race_history_dir, filename)
    df = pd.read_csv(raw_path)

    nulls_before = df.isnull().sum().sum()

    df_clean = clean_nulls(df)

    nulls_after = df_clean.isnull().sum().sum()

    processed_filename = filename.replace(".csv", ".parquet")
    processed_path = os.path.join(processed_dir, processed_filename)
    df_clean.to_parquet(processed_path, index=False)

    cleaning_summary.append({
        "file": filename,
        "rows": len(df_clean),
        "columns": len(df_clean.columns),
        "nulls_before": nulls_before,
        "nulls_after": nulls_after
    })

    print(f"Processed {filename}: {nulls_before} -> {nulls_after} nulls")

summary_df = pd.DataFrame(cleaning_summary)
summary_df

NameError: name 'os' is not defined